<a href="https://colab.research.google.com/github/DavidF09/CAP6545_Final_Project/blob/main/GEARS/GEARS_go_gnn_layers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## ***Packages***

In [ ]:
%pip install torch-geometric
%pip install scanpy
%pip install cell-gears

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 97.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 4.9 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 121.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 103.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## ***Imports and Loading Dataset***

In [ ]:
import torch
from gears import PertData, GEARS
import scanpy as sc
import sys

sys.path.append('../')
adata = sc.read("data/GSE90546/perturb_processed.h5ad")
print(adata)

AnnData object with n_obs × n_vars = 68603 × 5060
    obs: 'condition', 'cell_type', 'dose_val', 'control', 'condition_name'
    var: 'gene_name'
    uns: 'non_dropout_gene_idx', 'non_zeros_gene_idx', 'rank_genes_groups_cov_all', 'top_non_dropout_de_20', 'top_non_zero_de_20'


## ***Normalization & Subsetting 500 most variable genes***

In [ ]:
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata,n_top_genes=500, subset=True)

## ***Creating Dataloader***

In [ ]:
pert_data = PertData('./data')
pert_data.new_data_process(dataset_name = 'GSE90546', adata = adata)
pert_data.load(data_path = './data/GSE90546')
pert_data.prepare_split(split = 'simulation', seed = 1)
pert_data.get_dataloader(batch_size = 32, test_batch_size = 128)
print(pert_data.adata.obs["split"].value_counts())

Downloading...
100%|██████████| 9.46M/9.46M [00:00<00:00, 35.5MiB/s]
Downloading...
100%|██████████| 559k/559k [00:00<00:00, 5.27MiB/s]
Creating pyg object for each cell in the data...
Creating dataset file...
  9%|▉         | 8/87 [00:09<01:34,  1.20s/it]

SRPR+ctrl


 14%|█▍        | 12/87 [00:15<01:41,  1.35s/it]

SLMO2+ctrl


 16%|█▌        | 14/87 [00:17<01:29,  1.23s/it]

...

AMIGO3+ctrl


 67%|██████▋   | 58/87 [01:00<00:28,  1.01it/s]

KCTD16+ctrl


100%|██████████| 87/87 [01:26<00:00,  1.01it/s]
Done!
Saving new dataset pyg object at ./data/gse90546/data_pyg/cell_graphs.pkl
Done!
Found local copy...
These perturbations are not in the GO graph and their perturbation can thus not be predicted
['SRPR+ctrl' 'SLMO2+ctrl' 'TIMM23+ctrl' 'AMIGO3+ctrl' 'KCTD16+ctrl']
Creating pyg object for each cell in the data...
Creating dataset file...
100%|██████████| 82/82 [01:10<00:00,  1.17it/s]
Done!
Saving new dataset pyg object at ./data/GSE90546/data_pyg/cell_graphs.pkl
Done!
Creating new splits....
Saving new splits at ./data/GSE90546/splits/GSE90546_simulation_1_0.75.pkl
Simulation split test composition:
combo_seen0:0
combo_seen1:0
combo_seen2:0
unseen_single:21
Done!
Creating dataloaders....
Done!


split
train    52730
test      9741
val       3428
Name: count, dtype: int64


## ***Experiment One Go GNN layers = 2***

In [ ]:
gears_model = GEARS(pert_data, device = 'cuda:0',
                        weight_bias_track = False,
                        proj_name = 'pertnet',
                        exp_name = 'pertnet')
gears_model.model_initialize(hidden_size = 64,
                             num_go_gnn_layers = 2)

Downloading...
100%|██████████| 60.7M/60.7M [00:03<00:00, 18.3MiB/s]
Extracting tar file...
Done!


In [ ]:
gears_model.train(epochs = 7, lr = 1e-3)

Start Training...
Epoch 1 Step 1 Train Loss: 0.6396
Epoch 1 Step 51 Train Loss: 0.5154
Epoch 1 Step 101 Train Loss: 0.6011
Epoch 1 Step 151 Train Loss: 0.4595
Epoch 1 Step 201 Train Loss: 0.6349
Epoch 1 Step 251 Train Loss: 0.5136
Epoch 1 Step 301 Train Loss: 0.5501
Epoch 1 Step 351 Train Loss: 0.6341
Epoch 1 Step 401 Train Loss: 0.5194
Epoch 1 Step 451 Train Loss: 0.5080
Epoch 1 Step 501 Train Loss: 0.4902
Epoch 1 Step 551 Train Loss: 0.5414
Epoch 1 Step 601 Train Loss: 0.4849
Epoch 1 Step 651 Train Loss: 0.5462
Epoch 1 Step 701 Train Loss: 0.6191
Epoch 1 Step 751 Train Loss: 0.7082
Epoch 1 Step 801 Train Loss: 0.5593
Epoch 1 Step 851 Train Loss: 0.4841
Epoch 1 Step 901 Train Loss: 0.5684
Epoch 1 Step 951 Train Loss: 0.5163
Epoch 1 Step 1001 Train Loss: 0.5683
Epoch 1 Step 1051 Train Loss: 0.5389
Epoch 1 Step 1101 Train Loss: 0.5123
Epoch 1 Step 1151 Train Loss: 0.5825
Epoch 1 Step 1201 Train Loss: 0.5679
Epoch 1 Step 1251 Train Loss: 0.4999
Epoch 1 Step 1301 Train Loss: 0.5860
Epoch 

## ***Experiment Two Go GNN layers = 3***

In [ ]:
gears_model = GEARS(pert_data, device = 'cuda:0',
                        weight_bias_track = False,
                        proj_name = 'pertnet',
                        exp_name = 'pertnet')
gears_model.model_initialize(hidden_size = 64,
                             num_go_gnn_layers = 3)

Found local copy...


In [ ]:
gears_model.train(epochs = 7, lr = 1e-3)

Start Training...
Epoch 1 Step 1 Train Loss: 0.5138
Epoch 1 Step 51 Train Loss: 0.5617
Epoch 1 Step 101 Train Loss: 0.5261
Epoch 1 Step 151 Train Loss: 0.6576
Epoch 1 Step 201 Train Loss: 0.5267
Epoch 1 Step 251 Train Loss: 0.5512
Epoch 1 Step 301 Train Loss: 0.6895
Epoch 1 Step 351 Train Loss: 0.5729
Epoch 1 Step 401 Train Loss: 0.5157
Epoch 1 Step 451 Train Loss: 0.5973
Epoch 1 Step 501 Train Loss: 0.5946
Epoch 1 Step 551 Train Loss: 0.5499
Epoch 1 Step 601 Train Loss: 0.4965
Epoch 1 Step 651 Train Loss: 0.6496
Epoch 1 Step 701 Train Loss: 0.6373
Epoch 1 Step 751 Train Loss: 0.5908
Epoch 1 Step 801 Train Loss: 0.6389
Epoch 1 Step 851 Train Loss: 0.5259
Epoch 1 Step 901 Train Loss: 0.5045
Epoch 1 Step 951 Train Loss: 0.5164
Epoch 1 Step 1001 Train Loss: 0.6991
Epoch 1 Step 1051 Train Loss: 0.5406
Epoch 1 Step 1101 Train Loss: 0.5237
Epoch 1 Step 1151 Train Loss: 0.5104
Epoch 1 Step 1201 Train Loss: 0.5208
Epoch 1 Step 1251 Train Loss: 0.5514
Epoch 1 Step 1301 Train Loss: 0.4751
Epoch 

## ***Experiment Three Gene GNN layers = 2***

In [ ]:
gears_model = GEARS(pert_data, device = 'cuda:0',
                        weight_bias_track = False,
                        proj_name = 'pertnet',
                        exp_name = 'pertnet')
gears_model.model_initialize(hidden_size = 64,
                             num_gene_gnn_layers = 2)

Found local copy...


In [ ]:
gears_model.train(epochs = 7, lr = 1e-3)

Start Training...
Epoch 1 Step 1 Train Loss: 0.5592
Epoch 1 Step 51 Train Loss: 0.5388
Epoch 1 Step 101 Train Loss: 0.5467
Epoch 1 Step 151 Train Loss: 0.6151
Epoch 1 Step 201 Train Loss: 0.5195
Epoch 1 Step 251 Train Loss: 0.5387
Epoch 1 Step 301 Train Loss: 0.5654
Epoch 1 Step 351 Train Loss: 0.5025
Epoch 1 Step 401 Train Loss: 0.4588
Epoch 1 Step 451 Train Loss: 0.5522
Epoch 1 Step 501 Train Loss: 0.6476
Epoch 1 Step 551 Train Loss: 0.5552
Epoch 1 Step 601 Train Loss: 0.5053
Epoch 1 Step 651 Train Loss: 0.6018
Epoch 1 Step 701 Train Loss: 0.5746
Epoch 1 Step 751 Train Loss: 0.5133
Epoch 1 Step 801 Train Loss: 0.5803
Epoch 1 Step 851 Train Loss: 0.5125
Epoch 1 Step 901 Train Loss: 0.5169
Epoch 1 Step 951 Train Loss: 0.5725
Epoch 1 Step 1001 Train Loss: 0.5038
Epoch 1 Step 1051 Train Loss: 0.5033
Epoch 1 Step 1101 Train Loss: 0.5052
Epoch 1 Step 1151 Train Loss: 0.4947
Epoch 1 Step 1201 Train Loss: 0.5027
Epoch 1 Step 1251 Train Loss: 0.5016
Epoch 1 Step 1301 Train Loss: 0.5566
Epoch 

## ***Experiment Four Gene GNN layers = 3***

In [ ]:
gears_model = GEARS(pert_data, device = 'cuda:0',
                        weight_bias_track = False,
                        proj_name = 'pertnet',
                        exp_name = 'pertnet')
gears_model.model_initialize(hidden_size = 64,
                             num_gene_gnn_layers = 3)

Found local copy...


In [ ]:
gears_model.train(epochs = 7, lr = 1e-3)

Start Training...
Epoch 1 Step 1 Train Loss: 0.5636
Epoch 1 Step 51 Train Loss: 0.5407
Epoch 1 Step 101 Train Loss: 0.6424
Epoch 1 Step 151 Train Loss: 0.5335
Epoch 1 Step 201 Train Loss: 0.5329
Epoch 1 Step 251 Train Loss: 0.5456
Epoch 1 Step 301 Train Loss: 0.5673
Epoch 1 Step 351 Train Loss: 0.5237
Epoch 1 Step 401 Train Loss: 0.5950
Epoch 1 Step 451 Train Loss: 0.6222
Epoch 1 Step 501 Train Loss: 0.5057
Epoch 1 Step 551 Train Loss: 0.5877
Epoch 1 Step 601 Train Loss: 0.5252
Epoch 1 Step 651 Train Loss: 0.5407
Epoch 1 Step 701 Train Loss: 0.4782
Epoch 1 Step 751 Train Loss: 0.4851
Epoch 1 Step 801 Train Loss: 0.4494
Epoch 1 Step 851 Train Loss: 0.5937
Epoch 1 Step 901 Train Loss: 0.5665
Epoch 1 Step 951 Train Loss: 0.5909
Epoch 1 Step 1001 Train Loss: 0.6273
Epoch 1 Step 1051 Train Loss: 0.5469
Epoch 1 Step 1101 Train Loss: 0.6267
Epoch 1 Step 1151 Train Loss: 0.5532
Epoch 1 Step 1201 Train Loss: 0.6270
Epoch 1 Step 1251 Train Loss: 0.5052
Epoch 1 Step 1301 Train Loss: 0.5116
Epoch 